# Goldman Sachs India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** higher.gs.com

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 23:36:03
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Goldman_Sachs"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Goldman_Sachs/Outputs/2026_03_31


In [4]:
print("=" * 60)
print("GOLDMAN SACHS INDIA JOB SCRAPER")
print("Source: higher.gs.com (TAL.NET)")
print("=" * 60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def fetch_jd_selenium(driver, url, timeout=10):
    """Visit a job detail page and extract the JD text."""
    try:
        driver.get(url)
        time.sleep(random.uniform(2, 4))
        soup = BeautifulSoup(driver.page_source, "lxml")
        # Try common JD container selectors
        for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                     "[class*='details']", "article", "main", ".content"]:
            el = soup.select_one(sel)
            if el and len(el.get_text(strip=True)) > 100:
                return el.get_text(" ", strip=True)
        # Fallback: get body text
        body = soup.select_one("body")
        return body.get_text(" ", strip=True)[:5000] if body else ""
    except Exception as e:
        print(f"    [WARN] JD fetch failed for {url}: {e}")
        return ""

def fetch_jd_requests(session, url):
    """Fetch a job detail page via requests and extract JD text."""
    try:
        resp = session.get(url, timeout=20)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "lxml")
            for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                         "[class*='details']", "article", "main"]:
                el = soup.select_one(sel)
                if el and len(el.get_text(strip=True)) > 100:
                    return el.get_text(" ", strip=True)
            body = soup.select_one("body")
            return body.get_text(" ", strip=True)[:5000] if body else ""
    except:
        pass
    return ""


goldman_jobs = []

# Goldman uses higher.gs.com with Selenium
driver = setup_selenium()
try:
    # Try Bangalore as primary, then other India cities
    for city_query in ["Bengaluru%2C+India", "Mumbai%2C+India", "India"]:
        url = f"https://higher.gs.com/roles?location={city_query}"
        print(f"  Loading: {url}")
        driver.get(url)
        time.sleep(8)

        soup = BeautifulSoup(driver.page_source, "lxml")
        cards = soup.select("[class*='role-card'], [class*='job-card'], [class*='result'], a[href*='/roles/']")
        if not cards:
            cards = soup.select("[class*='opportunity'], [class*='position'], [class*='listing']")

        for card in cards:
            title_el = card.select_one("h2, h3, h4, [class*='title'], a")
            title = title_el.get_text(strip=True) if title_el else ""
            loc_el = card.select_one("[class*='location'], [class*='city']")
            loc = loc_el.get_text(strip=True) if loc_el else city_query.split("%2C")[0]

            link = card.select_one("a[href*='/roles/']")
            href = link.get("href", "") if link else ""

            if title and title not in [j["title"] for j in goldman_jobs]:
                goldman_jobs.append({
                    "job_id": href.split("/")[-1] if href else str(len(goldman_jobs)),
                    "title": title,
                    "company_name": "Goldman Sachs",
                    "raw_jd_text": card.get_text(" ", strip=True),
                    "location_city": loc.split(",")[0].strip(),
                    "industry": "Financial Services / Investment Banking",
                    "date_posted": datetime.now().strftime("%Y-%m-%d"),
                    "is_active": True,
                    "job_url": href if href.startswith("http") else f"https://higher.gs.com{href}" if href else "",
                    "business_unit": "",
                    "source_platform": "Selenium",
                })

        print(f"  {city_query}: {len(goldman_jobs)} total jobs so far")
        if len(goldman_jobs) >= 10:
            break

    # Fetch JDs for found jobs
    if goldman_jobs:
        print(f"  Fetching JD details for up to 30 jobs...")
        for i, job in enumerate(goldman_jobs[:30]):
            if job.get("raw_jd_text") and len(job["raw_jd_text"]) > 200:
                continue
            detail_url = f"https://higher.gs.com/roles/{job['job_id']}"
            jd = fetch_jd_selenium(driver, detail_url)
            if jd:
                goldman_jobs[i]["raw_jd_text"] = jd
            time.sleep(1)

except Exception as e:
    print(f"  Error: {e}")
finally:
    driver.quit()

print(f"Total Goldman Sachs India jobs: {len(goldman_jobs)}")


GOLDMAN SACHS INDIA JOB SCRAPER
Source: higher.gs.com (TAL.NET)


  Loading: https://higher.gs.com/roles?location=Bengaluru%2C+India


  Bengaluru%2C+India: 0 total jobs so far
  Loading: https://higher.gs.com/roles?location=Mumbai%2C+India


  Mumbai%2C+India: 0 total jobs so far
  Loading: https://higher.gs.com/roles?location=India


  India: 0 total jobs so far
Total Goldman Sachs India jobs: 0


In [5]:
df_goldman = save_results(goldman_jobs, "Goldman Sachs", OUTPUT_DIR)
if df_goldman is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_goldman.columns]
    print(df_goldman[cols].head(10).to_string())


  [WARN] No jobs found for Goldman Sachs
